> **Before running this**: this notebook needs things not included in the repo, a GGUF model in `app/models/` and embeddings in `app/datasets`. See [`../README.md` § Missing data](../README.md#missing-data-this-is-a-stripped-down-archive) and [`../app/README.md` § Setup and running](../app/README.md#setup-and-running) first.

# Agent pipeline over a real MCP connection

The project's tools (`get_time`, `get_schedule`, `list_keywords`, `search_pictograms`, `search_pictograms_by_synset`) live in `mcp_server/`, but `agent/agent.py` calls them as plain Python imports in the same process. Nothing goes through the MCP protocol at runtime.

This notebook runs the same tools through a real client/server exchange: the server starts as its own subprocess and every call goes out as JSON-RPC over stdio. 

Two turns run through the same pipeline, taken from the same row of `annotation/eval_final.parquet`: the row's `caregiver_clear` and `caregiver_vague` phrasings describe the same situation in two different ways. The decision step in `agent.py` is a probabilistic LLM call, so the clear phrasing usually skips the context tools and the vague one usually triggers them, but either can go the other way.

The row's `event_time`/`time_of_day` and `schedule` columns are injected into `get_time` and `get_schedule` via the `MOCK_TIME_INFO`/`MOCK_SCHEDULE_EVENTS` environment variables, read by `time_tool.py`/`schedule_tool.py` before falling back to the real clock or a real calendar. Both are pinned to the same row so they stay consistent. The calls still go through MCP exactly like any other tool call, only the data source changes.

## Setup

In [9]:
import asyncio
import sys
import time
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

PROJECT_ROOT = Path("..").resolve()
APP_DIR = PROJECT_ROOT / "app"
SRC_DIR = APP_DIR / "src"

assert (SRC_DIR / "mcp_server" / "server.py").exists()
print(SRC_DIR)


/Users/pelle/Development/GitHub/aac-mcp-agent/app/src


In [10]:
import json as _json
import random
from datetime import date, datetime

import pandas as pd

EVAL_PARQUET = PROJECT_ROOT / "annotation" / "eval_final.parquet"
assert EVAL_PARQUET.exists(), f"eval dataset not found: {EVAL_PARQUET}"

_eval_df = pd.read_parquet(EVAL_PARQUET)

# Pick one row at random, just to show if it works
EVAL_ROW = _eval_df.sample(1).iloc[0]

CLEAR_INPUT = EVAL_ROW["caregiver_clear"]
VAGUE_INPUT = EVAL_ROW["caregiver_vague"]
ROW_SCHEDULE = list(EVAL_ROW["schedule"])   # list[dict] shaped like ScheduleEvent

# Mock date time
_event_hhmm = datetime.strptime(EVAL_ROW["event_time"], "%H:%M").time()
_mock_dt = datetime.combine(date.today(), _event_hhmm)
ROW_TIME_INFO = {
    "current_dt": _mock_dt.isoformat(),
    "time_of_day": EVAL_ROW["time_of_day"],
}

print(f"sentence:        {EVAL_ROW['sentence']!r}")
print(f"caregiver_clear: {CLEAR_INPUT!r}")
print(f"caregiver_vague: {VAGUE_INPUT!r}")
print(f"mocked time:     {ROW_TIME_INFO['current_dt']}  ({ROW_TIME_INFO['time_of_day']})")
print(f"schedule events: {len(ROW_SCHEDULE)}")
for ev in ROW_SCHEDULE:
    print(f"  - {ev.get('start_time')}  {ev.get('title')}  @ {ev.get('location')}")

sentence:        'Can we use the tablet instead of the cards?'
caregiver_clear: 'He wants to use the tablet at 16:30 this afternoon'
caregiver_vague: 'he keeps reaching for the tablet'
mocked time:     2026-07-17T16:30:00  (afternoon)
schedule events: 1
  - 21:30  Audiologist check  @ hospital


In [11]:
import os

# Prepare the mocks
server_env = {
    **os.environ,
    "MOCK_TIME_INFO": _json.dumps(ROW_TIME_INFO),
    "MOCK_SCHEDULE_EVENTS": _json.dumps(ROW_SCHEDULE),
}

# Start the server
server_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server"],
    cwd=str(SRC_DIR),
    env=server_env,
)

## Load the real prompts, resolver, ranking, and the LLM backend

In [12]:
sys.path.insert(0, str(SRC_DIR))

from mcp_server.models import Pictogram, ScheduleEvent, TimeInfo
from agent.prompts import (
    build_decision_prompt,
    parse_decision_response,
    build_planner_prompt,
    build_planner_message,
    parse_new_planner_response,
)
from agent.context import build_context_block, filter_schedule_by_time
from agent.resolve import resolve_concept
from agent.ranking import rank_and_fill
from agent.backends import LlamaCppBackend
from config import AGENT_CANDIDATES_PER_TERM, AGENT_MAX_RESULTS, LANG

GGUF_PATH = APP_DIR / "models" / "Qwen2.5-3B-Instruct-Q4_K_M.gguf"
assert GGUF_PATH.exists(), f"model not found: {GGUF_PATH}"

backend = LlamaCppBackend(model_path=str(GGUF_PATH), n_ctx=2048)
print(f"backend: {GGUF_PATH.name}")


backend: Qwen2.5-3B-Instruct-Q4_K_M.gguf


## Pipeline function

Mirrors `AACAgent.run()` phase by phase. Decide and plan are plain LLM calls, nothing changes there. Context and retrieval are where `agent.py` calls tool functions directly. here those calls go through `session.call_tool()` instead.

`list_tools()` runs once at connection time, the way a real MCP client would probe the server first.

In [13]:
def _pic_from_mcp(d: dict) -> Pictogram:
    return Pictogram.model_validate(d)


async def run_turn_via_mcp(session: ClientSession, raw_input: str, kw_set: set[str], lang: str = LANG):
    trace = {"raw_input": raw_input}

    #### Phase 1: decide (LLM only, no tool) ################################################################################################
    system_msg = build_decision_prompt()
    user_msg   = build_planner_message(raw_input, history="")
    raw_text   = backend.chat(system_msg, user_msg)
    needs_context = bool(parse_decision_response(raw_text).get("needs_context", True))
    trace["needs_context"] = needs_context

    #### Phase 2: context tools via MCP, only if needed #####################################################################################
    context_block = ""
    tool_calls_done = []
    if needs_context:
        time_result = await session.call_tool("get_time", {})
        tool_calls_done.append("get_time")
        time_payload = json.loads(time_result.content[0].text)
        time_info = TimeInfo.model_validate(time_payload)
        exact_time = time_info.current_dt.strftime("%H:%M")
        time_of_day = time_info.time_of_day
        schedule_events: list[ScheduleEvent] = []
        try:
            sched_result = await session.call_tool("get_schedule", {})
            sched_payload = []
            for block in sched_result.content:
                item = json.loads(block.text)
                sched_payload.extend(item) if isinstance(item, list) else sched_payload.append(item)
            schedule_events = [ScheduleEvent.model_validate(e) for e in sched_payload]
            if schedule_events:
                tool_calls_done.append("get_schedule")
        except Exception as exc:
            print(f"get_schedule failed (no calendar configured?): {exc}")

        if schedule_events:
            relevant = filter_schedule_by_time(schedule_events, time_of_day)
            context_block = build_context_block(time_of_day, relevant, exact_time=exact_time)
        else:
            context_block = build_context_block(time_of_day, [], exact_time=exact_time)
    trace["tool_calls"] = tool_calls_done
    trace["context_block"] = context_block

    #### Phase 3: plan (LLM only, no tool) ################################################################################################
    system_msg = build_planner_prompt(full=False)
    user_msg   = build_planner_message(raw_input, history="", context_block=context_block)
    raw_text   = backend.chat(system_msg, user_msg)
    parsed     = parse_new_planner_response(raw_text)
    concepts   = list(dict.fromkeys(str(c).strip() for c in parsed.get("concepts", []) if c))
    trace["concepts"] = concepts

    #### Phase 4: retrieval via MCP #######################################################################################################
    seen_ids: set[int] = set()
    seen_queries: set[str] = set()
    candidates: list[Pictogram] = []
    concept_order: dict[int, int] = {}
    for concept_idx, term in enumerate(concepts):
        queries, method = resolve_concept(term, kw_set, lang=lang, return_method=True)
        if not queries:
            continue
        for query in queries:
            if query in seen_queries:
                continue
            seen_queries.add(query)
            result = await session.call_tool(
                "search_pictograms",
                {"keyword": query, "lang": lang, "max_results": AGENT_CANDIDATES_PER_TERM},
            )
            payload = json.loads(result.content[0].text)
            for pic_dict in payload.get("results", []):
                pic = _pic_from_mcp(pic_dict)
                if pic.id not in seen_ids:
                    seen_ids.add(pic.id)
                    concept_order[pic.id] = concept_idx
                    candidates.append(pic)
    trace["n_candidates"] = len(candidates)

    #### Phase 5: rank (local, deterministic) ################################################################################################
    window, pool_ids = rank_and_fill(candidates, set(), concept_order, AGENT_MAX_RESULTS, "sequential_blocks")
    trace["result"] = [(p.id, p.keywords[0] if p.keywords else "?") for p in window]

    return trace

## Load the keyword set

In [14]:
import json

async def _load_kw_set() -> set[str]:
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as s:
            await s.initialize()
            raw = await s.call_tool("list_keywords", {"lang": LANG})
            payload = json.loads(raw.content[0].text)
            return set(payload.get("keywords", []))


kw_set = await _load_kw_set()
print(f"{len(kw_set)} keywords loaded, lang={LANG!r}")


15718 keywords loaded, lang='en'


## Connect, run both turns, and shut down

Everything runs inside one `async` function so the connection opens and closes in the same task. `stdio_client`/`ClientSession` use anyio cancel scopes tied to the task that entered them, and splitting open/close across separate cells can hand the exit call to a different task, raising `RuntimeError` on shutdown.

In [15]:
from mcp_server.tools.arasaac import search_pictograms as search_pictograms_direct

SAMPLE_KEYWORD = "I" # just to see the times

import statistics

async def run_both_turns():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_response = await session.list_tools()
            print("tools:", [t.name for t in tools_response.tools])

            # warm-up BOTH sides before anything else touches the process
            # search_pictograms_direct(keyword=SAMPLE_KEYWORD, lang=LANG, max_results=5)
            # await session.call_tool(
            #     "search_pictograms", {"keyword": SAMPLE_KEYWORD, "lang": LANG, "max_results": 5}
            # )

            clear_trace = await run_turn_via_mcp(session, CLEAR_INPUT, kw_set)
            vague_trace = await run_turn_via_mcp(session, VAGUE_INPUT, kw_set)

            # now measure, repeated, take median
            direct_times = []
            mcp_times = []
            for _ in range(20):
                t0 = time.perf_counter()
                search_pictograms_direct(keyword=SAMPLE_KEYWORD, lang=LANG, max_results=5)
                direct_times.append(time.perf_counter() - t0)

                t0 = time.perf_counter()
                await session.call_tool(
                    "search_pictograms", {"keyword": SAMPLE_KEYWORD, "lang": LANG, "max_results": 5}
                )
                mcp_times.append(time.perf_counter() - t0)
            t_direct = statistics.median(direct_times)
            t_mcp = statistics.median(mcp_times)

    return clear_trace, vague_trace, t_direct, t_mcp

In [16]:
clear_trace, vague_trace, t_direct, t_mcp = await run_both_turns()

print("\n=== clear ===")
print(clear_trace["raw_input"])
print("needs_context:", clear_trace["needs_context"])
print("tool_calls:", clear_trace["tool_calls"])
print("concepts:", clear_trace["concepts"])
print("candidates:", clear_trace["n_candidates"])
print("result:", clear_trace["result"])

print("\n=== vague ===")
print(vague_trace["raw_input"])
print("needs_context:", vague_trace["needs_context"])
print("tool_calls:", vague_trace["tool_calls"])
print("context_block:", vague_trace["context_block"])
print("concepts:", vague_trace["concepts"])
print("candidates:", vague_trace["n_candidates"])
print("result:", vague_trace["result"])

print("\n=== timing: direct call vs MCP call ===")
print(f"direct import: {t_direct*1000:.2f} ms")
print(f"MCP call_tool: {t_mcp*1000:.2f} ms")
print(f"overhead: {(t_mcp - t_direct)*1000:.2f} ms")

tools: ['search_pictograms', 'get_pictogram_metadata', 'list_keywords', 'search_pictograms_by_synset', 'get_time', 'get_schedule']


[07/17/26 15:34:26] INFO     Loading GGUF model from                                                ]8;id=9028439;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/backends.py\backends.py]8;;\:]8;id=9028440;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/backends.py#178\178]8;;\
                             '/Users/pelle/Development/GitHub/aac-mcp-agent/app/models/Qwen2.5-3B-I                
                             nstruct-Q4_K_M.gguf' (n_ctx=2048) ...                                                 

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[07/17/26 15:34:30] INFO     GGUF model loaded.                                                     ]8;id=9028445;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/backends.py\backends.py]8;;\:]8;id=9028446;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/backends.py#180\180]8;;\

[07/17/26 15:34:47] INFO     [LLAMACPP] prompt_tokens=241 completion_tokens=6                       ]8;id=9028451;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/backends.py\backends.py]8;;\:]8;id=9028452;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/agent/backends.py#194\194]8;;\

...

                    INFO     search_pictograms('I', lang='en'): 5 results from local dataset.        ]8;id=9028579;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=9028580;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#218\218]8;;\

                    INFO     search_pictograms('I', lang='en'): 5 results from local dataset.        ]8;id=9028585;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=9028586;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#218\218]8;;\

                    INFO     search_pictograms('I', lang='en'): 5 results from local dataset.        ]8;id=9028591;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=9028592;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#218\218]8;;\

                    INFO     search_pictograms('I', lang='en'): 5 results from local dataset.        ]8;id=9028597;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py\arasaac.py]8;;\:]8;id=9028598;file:///Users/pelle/Development/GitHub/aac-mcp-agent/app/src/mcp_server/tools/arasaac.py#218\218]8;;\


=== clear ===
He wants to use the tablet at 16:30 this afternoon
needs_context: False
tool_calls: []
concepts: ['I', 'tablet', 'use', 'afternoon', 'time', 'play', 'screen', 'fun', 'day', 'press']
candidates: 42
result: [(6632, Keyword(type=1, keyword='I', plural=None, meaning='pron. Shape pron. pers. com. first person singular, as in the sentence serves as the subject.')), (2617, Keyword(type=1, keyword='I', plural=None, meaning='pron. Shape pron. pers. com. first person singular, as in the sentence serves as the subject.')), (3030, Keyword(type=6, keyword='i', plural=None, meaning='f. Tenth letter of the alphabet Spanish and Latino ninth international order, which is a closed vowel sound palate.')), (3117, Keyword(type=6, keyword='I', plural=None, meaning='f. Tenth letter of the alphabet Spanish and Latino ninth international order, which is a closed vowel sound palate.')), (31807, Keyword(type=1, keyword='I', plural=None, meaning='pron. Shape pron. pers. com. first person singular, 

## Notes

As we can see we have a small latency (in proportion it is doubled actually) passing from normal to mcp.